In [25]:
import sys

sys.path.append("01-agentic-rag/")
sys.path.append("04-evaluation/")
from ingest import load_faq_data
documents = load_faq_data()

In [18]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [13]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)
documents = documents_llm

In [15]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [16]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [19]:
import json

user_prompt = json.dumps(doc)

In [ ]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

In [21]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [22]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [23]:
result = response.output_parsed

print(result)

questions=['Can I still join the course if I only found out about it now?', 'Is it too late to start this course, or can new students still jump in?', 'If I join after the course has already started, am I still eligible for a certificate?', 'What do I need to do if I want a certificate but I’m joining late?', 'Can I enroll now and still get certified as long as I finish my project on time?']


In [30]:
from evaluation_utils import llm_structured, calc_price

In [27]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['I just found this course — is it still okay to jump in now?', 'Can I join the course late, or is it too late already?', 'If I start the course now, can I still get a certificate somehow?', 'What do I need to do to be eligible for the certificate if I joined late?', 'Is there a deadline for submitting the project if I want the certificate?']


In [28]:
usage.input_tokens, usage.output_tokens

(207, 91)

In [31]:
cost = calc_price(usage)

cost

{'input_cost': 0.00015525,
 'output_cost': 0.00040950000000000003,
 'total_cost': 0.00056475}

In [32]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'I just found this course — is it still okay to jump in now?',
  'document': '74eb249bbf'},
 {'question': 'Can I join the course late, or is it too late already?',
  'document': '74eb249bbf'},
 {'question': 'If I start the course now, can I still get a certificate somehow?',
  'document': '74eb249bbf'},
 {'question': 'What do I need to do to be eligible for the certificate if I joined late?',
  'document': '74eb249bbf'},
 {'question': 'Is there a deadline for submitting the project if I want the certificate?',
  'document': '74eb249bbf'}]

## 03-ground-truth-batch

In [33]:
from evaluation_utils import llm_structured_retry

In [34]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [35]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [36]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [37]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/1400 [00:00<?, ?it/s]

In [38]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

7000

In [39]:
from evaluation_utils import calc_price, calc_total_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

1.0910010000000003

In [40]:
calc_total_price(usages)

1.0910010000000003

In [41]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

In [43]:
df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)